In [1]:
import os

# Ensure protobuf fallback (kept from original script)
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

# Set reproducibility seeds
np.random.seed(42)

BASE_DIR = "/kaggle/working"
TRAIN_PATH = os.path.join(BASE_DIR, "data", "train.csv")
TEST_PATH = os.path.join(BASE_DIR, "data", "test.csv")
SUBMISSION_PATH = os.path.join(BASE_DIR, "submission.csv")

DATASET_SIZE = 300000  # number of rows to read from train.csv
LEARNING_RATE = 1e-3  # kept for compatibility, not used by RF
EPOCHS = 12  # kept for compatibility, not used by RF
BATCH_SIZE = 1024  # kept for compatibility, not used by RF
RANDOM_STATE = 42


def clean(df):
    """Drop rows with NaN and ensure passenger count is positive."""
    df = df.dropna()
    df = df[df["passenger_count"] > 0]
    return df.reset_index(drop=True)


def add_time_features(df):
    """Extract hour, weekday, month from pickup_datetime."""
    dt = pd.to_datetime(df["pickup_datetime"])
    df["pickup_hour"] = dt.dt.hour
    df["pickup_weekday"] = dt.dt.weekday
    df["pickup_month"] = dt.dt.month
    return df


def haversine_distance(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance (km)."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


def add_distance_features(df):
    """Add haversine distance column."""
    df["haversine_km"] = haversine_distance(
        df["pickup_latitude"],
        df["pickup_longitude"],
        df["dropoff_latitude"],
        df["dropoff_longitude"],
    )
    return df


def output_submission(test_df, predictions, key_col, target_col, file_name):
    """Write predictions together with the key column to a CSV file."""
    pred = predictions.squeeze()
    out_df = pd.DataFrame({key_col: test_df[key_col], target_col: pred})
    out_df.to_csv(file_name, index=False)
    print(f"Submission written to {file_name}")


# Define column dtypes for efficient loading
datatypes = {
    "key": "str",
    "fare_amount": "float32",
    "pickup_datetime": "str",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "uint8",
}

# Load a subset of the training data
train_df = pd.read_csv(
    TRAIN_PATH,
    nrows=DATASET_SIZE,
    dtype=datatypes,
    usecols=list(datatypes.keys()),
)

# Load the test data
test_df = pd.read_csv(
    TEST_PATH,
    dtype={k: v for k, v in datatypes.items() if k != "fare_amount"},
    usecols=[
        "key",
        "pickup_datetime",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
    ],
)

# Basic cleaning and feature engineering
train_df = clean(train_df)
train_df = add_time_features(train_df)
train_df = add_distance_features(train_df)

test_df = add_time_features(test_df)
test_df = add_distance_features(test_df)

# Remove raw datetime column (non‑numeric)
train_df = train_df.drop(columns=["pickup_datetime"])
test_df = test_df.drop(columns=["pickup_datetime"])

target_col = "fare_amount"
feature_cols = [c for c in train_df.columns if c not in ["key", target_col]]

# Scale features to [0, 1] – helpful for many models
scaler = MinMaxScaler()
train_features = scaler.fit_transform(train_df[feature_cols])
test_features = scaler.transform(test_df[feature_cols])

y = train_df[target_col].values

# Train/validation split (validation not used by RF but kept for parity)
X_train, X_val, y_train, y_val = train_test_split(
    train_features, y, test_size=0.2, random_state=RANDOM_STATE
)



FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/data/train.csv'

In [2]:
# Random Forest model – replaces the original Keras network
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# Fit on the training portion
rf_model.fit(X_train, y_train)



NameError: name 'X_train' is not defined

In [3]:
# Predict on the test set and write submission
test_predictions = rf_model.predict(test_features)
output_submission(
    test_df,
    test_predictions,
    key_col="key",
    target_col="fare_amount",
    file_name=SUBMISSION_PATH,
)

NameError: name 'test_features' is not defined